# SINDy Paper Implementation v1

---
Simple one variable systems
- Implementing SINDy for simple one variable systems


In [60]:
import numpy as np

### Library construction




In [83]:
def createLibrary(x):
  # 1 X X^2 X^3
  PHI = np.zeros((len(x),4))
  print(PHI.size)
  PHI[:,0] = 1
  PHI[:,1] = x
  PHI[:,2] = x**2
  PHI[:,3] = x**3
  return PHI

### Models

### 1. Model 1 : Linear decay

$$
\dot{x} = -2x
$$

Solution:

$$
x(t) = x_0 e^{-2t}
$$

---

### 2. Model 2 : Quadratic decay

$$
\dot{x} = -x^2
$$

Solution:

$$
x(t) = \frac{x_0}{1 + x_0 t}
$$

---

### 3. Model 3: Cubic nonlinear system

$$
\dot{x} = x - x^3
$$

Solution:

$$
x(t) =
\frac{x_0}
{\sqrt{x_0^2 + \left(1-x_0^2\right)e^{-2t}}}
$$

---

### 4. Model 4: Simple harmonic oscillator

$$
\dot{x} = y
$$

$$
\dot{y} = -4x
$$

Solution:

$$
x(t) = A\cos(2t)
$$

$$
y(t) = -2A\sin(2t)
$$
sampling interval $T_s = 0.01s$,
number of samples = 10000(?)

In [101]:
# dx/dt = -ax
def Model1(const, init_condition,t_int):
  x = init_condition*np.exp(-const*t_int)
  x_dot = -const*init_condition*np.exp(-const*t_int)
  return x,x_dot

def Model2(init_condition,t_int):
  x = init_condition/(1+init_condition*t_int)
  x_dot = -x**2
  return x,x_dot

def Model3(init_condition, t_int):
  x = init_condition/(np.sqrt(init_condition**2 + (1-init_condition**2)*np.exp(-2*t_int)))
  x_dot = x - x**3
  return x,x_dot

In [95]:
# sampling interval
T_s = 0.01
start = 0
end = 20

t_int = np.arange(start, end, T_s)

# initial conditions
x_dot = np.zeros(len(t_int))

x, x_dot = Model1(3, 2, t_int)

[-6.00000000e+00 -5.82267320e+00 -5.65058720e+00 ... -5.74868933e-26
 -5.57878989e-26 -5.41391173e-26]


In [96]:
PHI = createLibrary(x)
PHI

8000


array([[1.00000000e+00, 2.00000000e+00, 4.00000000e+00, 8.00000000e+00],
       [1.00000000e+00, 1.94089107e+00, 3.76705813e+00, 7.31144948e+00],
       [1.00000000e+00, 1.88352907e+00, 3.54768175e+00, 6.68216169e+00],
       ...,
       [1.00000000e+00, 1.91622978e-26, 3.67193656e-52, 7.03627419e-78],
       [1.00000000e+00, 1.85959663e-26, 3.45809962e-52, 6.43067041e-78],
       [1.00000000e+00, 1.80463724e-26, 3.25671558e-52, 5.87719023e-78]])

### Parameters

In [97]:
lamb = 0.1
n_iterations = 10

### Regression

In [108]:
def SparseRegression(PHI, x_dot, n_iterations, lamb):
  ## Ordinary Least Squares
  # E - coefficient vector
  E,*_ = np.linalg.lstsq(PHI, x_dot, rcond = None)
  print(E)
  ### iterations
  for i in range(n_iterations):
    ## getting the idices |e_j| < lambda
    smallIndices = np.abs(E)<lamb
    ## setting small coefficients to 0
    E[smallIndices] = 0
    ## doing linear regression with the remaining coefficients
    E[~smallIndices],*_ = np.linalg.lstsq(PHI[:,~smallIndices], x_dot, rcond = None)
  return E

In [121]:
### Model 1
lamb = 0.1
n_iterations = 10
E1 = SparseRegression(PHI, x_dot, n_iterations, lamb)
E1

[ 4.32230389e-16 -3.00000000e+00  3.66373598e-14 -1.95399252e-14]


array([ 0., -3.,  0.,  0.])

In [124]:
### Model 2
lamb = 0.1
n_iterations = 10
x_2, x_dot2 = Model2(4,t_int)
PHI2 = createLibrary(x_2)

E2 = SparseRegression(PHI2, x_dot2, n_iterations, lamb)
E2

8000
[ 7.66037603e-16 -5.66213743e-15 -1.00000000e+00 -2.77555756e-16]


array([ 0.,  0., -1.,  0.])

In [125]:
#### Model 3
lamb = 0.1
n_iterations = 10
x_3, x_dot3 = Model3(6,t_int)
PHI3 = createLibrary(x_3)

E3 = SparseRegression(PHI3, x_dot3, n_iterations, lamb)
E3

8000
[-2.16543892e-14  1.00000000e+00 -1.23234756e-14 -1.00000000e+00]


array([ 0.,  1.,  0., -1.])

###